# ROB 450: compile our fine-tuned turkey model for the Hailo-8L

Makes `f26_turkey_v1.hef` (set `NAME` in step 2; F26 = our team, bump v1 to v2 per retrain): the fine-tuned YOLO26n cut at its six head convolutions (box + class per stride), the layout of W26's working `turkey_640_v4.hef`. The robot's `Vision/Detector.py` and `decode_split` run it with no code change.

**Before you run:** Runtime > Change runtime type > **T4 GPU** (quantization is much better and faster with a GPU).

**Drive folder** `My Drive/ROB450 Capstone Turkey Deterrence/hailo/` must hold:
- `hailo_dataflow_compiler-*.whl` (**3.33.0 matches the Pi's HailoRT 4.23**; a 3.34 HEF may not load on the Pi)
- `NAME.onnx`, `calib_NAME.zip`, `eval_NAME.zip`, `yolo_decode.py` plus `SHA256.txt` and the `.part-*` files, all built by `python -I Vision/training/hailo_kit.py CV_DIR best.pt OUT_DIR`

Run all cells in order. Step 6 is the gate: if it says FAIL, stop and paste its output back to Claude instead of compiling.

In [ ]:
# 1. Machine check: expect x86_64 and a GPU
!uname -m
!nvidia-smi -L || echo "NO GPU: switch the runtime to T4 GPU for better quantization"


In [ ]:
# 2. Connect Drive and find the files
from google.colab import drive
drive.mount('/content/drive')
import glob, os, re, shutil
cands = glob.glob('/content/drive/MyDrive/ROB450 Capstone Turkey Deterrence/hailo*')
assert cands, 'No hailo folder under My Drive/ROB450 Capstone Turkey Deterrence/'
D = cands[0]; print('folder:', repr(D))
NAME = 'f26_turkey_v1'   # the model's name; the kit (hailo_kit.py --name) must match
os.environ['NAME'] = NAME
need = [f'{NAME}.onnx', f'calib_{NAME}.zip', f'eval_{NAME}.zip', 'yolo_decode.py']
# NAME.onnx and calib_NAME.zip arrive split into .part-aa, .part-ab, ... (10 MB upload limit)
# hashes come from SHA256.txt, written by Vision/training/hailo_kit.py next to the parts
SHA = {}
if os.path.exists(f'{D}/SHA256.txt'):
    for line in open(f'{D}/SHA256.txt'):
        h, f = line.split()
        SHA[f] = h
print('hashes for', sorted(SHA) or 'none (SHA256.txt missing: rejoined files are not checked)')
missing = [f for f in need if not os.path.exists(f'{D}/{f}') and not glob.glob(f'{D}/{f}.part-*')]
assert not missing, f'Upload these to the Drive hailo folder first: {missing}'
whl = sorted(glob.glob(f'{D}/hailo_dataflow_compiler-*.whl'))
assert whl, 'Put the Hailo Dataflow Compiler .whl in the hailo folder'
WHL = whl[0] if any('3.33' in w for w in whl) is False else [w for w in whl if '3.33' in w][0]
DFC = re.search(r'compiler-(\d+\.\d+)', WHL).group(1)
tag = re.search(r'-cp(\d)(\d+)-', WHL)
PY = f'{tag.group(1)}.{tag.group(2)}' if tag else '3.10'
print('compiler', DFC, '| python', PY, '|', os.path.basename(WHL))
if DFC != '3.33':
    print('WARNING: the Pi runs HailoRT 4.23, which pairs with compiler 3.33. A', DFC,
          'HEF may refuse to load on the Pi. Continue to test it, but get 3.33 if it fails.')


In [ ]:
# 3. Install the compiler in its own Python
!apt-get -qq update && apt-get -qq install -y graphviz graphviz-dev > /dev/null
!pip -q install uv
!uv python install {PY}
!uv venv -q -p {PY} /content/hvenv
!uv pip install -q -p /content/hvenv/bin/python "{WHL}" opencv-python-headless
!/content/hvenv/bin/python -c "import hailo_sdk_client as h; print('DFC', h.__version__)"


In [ ]:
# 4. Copy the model and images to the Colab machine (rejoin split files, check their hashes)
import hashlib
os.makedirs('/content/work/calib', exist_ok=True); os.makedirs('/content/work/eval', exist_ok=True)
for f in need:
    parts = sorted(glob.glob(f'{D}/{f}.part-*'))
    if parts:
        with open(f'/content/work/{f}', 'wb') as out:
            for p in parts: out.write(open(p, 'rb').read())
    else:
        shutil.copy(f'{D}/{f}', '/content/work/')
    if f in SHA:
        h = hashlib.sha256(open(f'/content/work/{f}', 'rb').read()).hexdigest()
        assert h == SHA[f], f'{f} is corrupt after rejoining ({len(parts)} parts): {h}'
        print(f, 'rejoined from', len(parts), 'parts, hash OK')
!unzip -q -o /content/work/calib_{NAME}.zip -d /content/work/calib
!unzip -q -o /content/work/eval_{NAME}.zip -d /content/work/eval
!ls /content/work/calib | wc -l; ls /content/work/eval | wc -l


In [ ]:
%%writefile /content/work/step5.py
# 5. Translate (split heads, /255 inside the model) and quantize with the calibration images
import os
NAME = os.environ["NAME"]
import glob, numpy as np, cv2
from hailo_sdk_client import ClientRunner
ENDS = [f"/model.23/cv{k}.{i}/cv{k}.{i}.2/Conv" for i in range(3) for k in (2, 3)]
runner = ClientRunner(hw_arch="hailo8l")
runner.translate_onnx_model(f"{NAME}.onnx", NAME, start_node_names=["images"],
                            end_node_names=ENDS, net_input_shapes={"images": [1, 3, 640, 640]})
# the robot sends uint8 RGB 0-255 (letterboxed, pad 114); the model was trained on 0-1
runner.load_model_script("normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])\n")
def load(folder):
    fs = sorted(glob.glob(f"{folder}/*.jpg"))
    return np.stack([cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB) for f in fs]).astype(np.float32), fs
calib, _ = load("calib")
print("calibration set", calib.shape)
runner.optimize(calib)
runner.save_har(f"{NAME}_quantized.har")
print("saved", f"{NAME}_quantized.har")


In [ ]:
%cd /content/work
!/content/hvenv/bin/python step5.py 2>&1 | grep -v -E "^\\s*$" | tail -25


In [ ]:
%%writefile /content/work/step6.py
# 6. GATE: does the quantized model still find what the float model finds? (W26's HEFs failed exactly here)
import os
NAME = os.environ["NAME"]
import glob, numpy as np, cv2
from hailo_sdk_client import ClientRunner, InferenceContext
from yolo_decode import decode_split
runner = ClientRunner(har=f"{NAME}_quantized.har")
fs = sorted(glob.glob("eval/*.jpg"))
x = np.stack([cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB) for f in fs]).astype(np.float32)
res = {}
for name, ctx_t in (("float", InferenceContext.SDK_FP_OPTIMIZED), ("quant", InferenceContext.SDK_QUANTIZED)):
    with runner.infer_context(ctx_t) as ctx:
        out = runner.infer(ctx, x)
    out = out if isinstance(out, list) else [out]
    res[name] = out
    print(name, "outputs", [o.shape for o in out])
def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x2 - x1) * max(0, y2 - y1)
    return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)
nf = nq = matched = 0; ious = []; top_f = []; top_q = []
for j in range(len(fs)):
    bf, sf, cf = decode_split([o[j] for o in res["float"]], (640, 640), 0.25)
    bq, sq, cq = decode_split([o[j] for o in res["quant"]], (640, 640), 0.25)
    nf += len(bf); nq += len(bq)
    top_f.append(sf.max() if len(sf) else 0); top_q.append(sq.max() if len(sq) else 0)
    for b in bf:
        best = max([iou(b, q) for q in bq], default=0)
        if best > 0.7: matched += 1; ious.append(best)
recall = matched / max(nf, 1)
print(f"float boxes {nf}, quantized boxes {nq}, float boxes kept {matched} ({recall:.0%}), mean IoU {np.mean(ious) if ious else 0:.3f}")
print(f"mean top score float {np.mean(top_f):.3f}, quantized {np.mean(top_q):.3f}")
ok = nf > 0 and recall >= 0.85 and np.mean(top_q) > 0.5 * np.mean(top_f)
print("GATE:", "PASS, go on to step 7" if ok else "FAIL: do not compile; paste this output to Claude")


In [ ]:
%cd /content/work
!/content/hvenv/bin/python step6.py 2>&1 | grep -v -E "^\\s*$" | tail -12


In [ ]:
%%writefile /content/work/step7.py
# 7. Compile for the Hailo-8L and save the HEF, HAR and log to Drive
from hailo_sdk_client import os
NAME = os.environ["NAME"]
import ClientRunner
runner = ClientRunner(har=f"{NAME}_quantized.har")
hef = runner.compile()
open(f"{NAME}.hef", "wb").write(hef)
print("wrote", f"{NAME}.hef", len(hef) // 1024, "KB")


In [ ]:
%cd /content/work
!/content/hvenv/bin/python step7.py 2>&1 | tee compile.log | grep -v -E "^\\s*$" | tail -15
!cp {NAME}.hef {NAME}_quantized.har compile.log "{D}/"
!ls -la "{D}" | grep -E "{NAME}|compile.log"


Done. Tell Claude the HEF is in Drive (or download `f26_turkey_v1.hef` to the Mac's Downloads). Claude copies it to `~/ME450/Sandbox/hef/` on the Pi, checks it loads on HailoRT 4.23, compares it to the `.pt`, and runs the robot with `VISION_HEF_PATH` pointing at it.